# Z2005 — Week 0: Object-Oriented Programming, Then Algorithms
A self-study notebook covering Python classes (encapsulation, inheritance, polymorphism) and the first tools for reasoning about running time (Big-O).

## Learning Objectives

By the end of this notebook you will be able to:

- Write a Python class with a constructor, attributes, and methods, and explain what `self` refers to.
- Apply encapsulation to refuse invalid state changes instead of allowing them.
- Use inheritance (`class Child(Parent):`) to express genuine "is-a" relationships, and distinguish that from composition ("has-a").
- Explain polymorphism and write code that treats different object types uniformly through a shared interface.
- Classify a piece of code as O(1), O(log n), O(n), O(n log n), or O(n²), and justify the classification.
- Spot the single most common bug in a first `__init__` (forgetting `self.`).

## How to use this notebook

Run the cells from top to bottom. Markdown cells explain a concept before any code appears; code cells are heavily commented and runnable as-is.

Cells marked `# TODO: implement this` are for you to complete — they raise `NotImplementedError` until you do. Cells with `assert` statements are self-checks: they raise an `AssertionError` (or print an error) if your code is wrong, and print a friendly success message if it is right. Solutions are collected at the very end — try each exercise yourself first.

## 1. A class is a blueprint

Before classes, you might track a point on a plane as a plain tuple: `p1 = (0, 0)`. That works, but the tuple says nothing about *what the numbers mean* — is `p1[0]` an x-coordinate, a weight, an index? A **class** is a blueprint for building objects that carry both data (**attributes**) and behavior (**methods**) together, with names attached to each piece.

`__init__` runs exactly once, automatically, whenever you create a new object with `Point(0, 0)`. Inside every method, `self` refers to the *specific* object the method was called on — it is passed automatically by Python, you never write it yourself when calling a method (only when defining one).

**Common pitfall:** writing `x = x` instead of `self.x = x` inside `__init__`. This assigns to the local parameter and throws the value away — the object never actually gets an `x` attribute, and later code fails with `AttributeError`, often far from where the real mistake was made.

In [ ]:
class Point:
    """A point in the 2D plane."""

    def __init__(self, x, y):
        # runs once, automatically, when Point(x, y) is called
        self.x = x   # self.x is THIS point's x -- not a local/global variable named x
        self.y = y

    def distance_to(self, other):
        # self is the point distance_to was called ON; other is the argument
        dx = self.x - other.x
        dy = self.y - other.y
        return (dx ** 2 + dy ** 2) ** 0.5

    def midpoint(self, other):
        # returns a brand-new Point sitting exactly between self and other
        return Point((self.x + other.x) / 2, (self.y + other.y) / 2)


p1 = Point(0, 0)
p2 = Point(3, 4)
print(p1.distance_to(p2))          # 5.0 -- the classic 3-4-5 triangle
mid = p1.midpoint(p2)
print(mid.x, mid.y)                # 1.5 2.0

**The bug almost everyone hits once**, shown deliberately so you recognize it instantly later:

In [ ]:
class BrokenPoint:
    def __init__(self, x, y):
        x = x   # BUG: assigns to the local parameter, self.x is never set
        y = y

try:
    bp = BrokenPoint(3, 4)
    print(bp.x)   # never reached
except AttributeError as e:
    print(f"Caught the classic mistake: {e}")

## 2. Encapsulation

With `Point` above, nothing stops `p1.x = "banana"` — attributes can be read and rewritten freely, and the resulting crash (when some later method tries arithmetic on a string) happens far from the actual mistake. **Encapsulation** means routing every state change through the object's own methods, so invalid states are refused at the door rather than merely being unlikely.

Python has no `private` keyword. A leading underscore (`self._balance`) is a *convention*, not a lock: it tells other programmers "go through the methods, not the attribute directly" — Python will not stop `account._balance = -500`, but a careful codebase never does that. Encapsulation is a social and structural contract, not secrecy: anyone reading the source still sees `_balance`.

In [ ]:
class BankAccount:
    def __init__(self, balance=0):
        self._balance = balance   # leading underscore: "do not touch directly"

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self._balance += amount

    def withdraw(self, amount):
        # checked BEFORE the subtraction runs -- an invalid balance is never reachable
        if amount > self._balance:
            raise ValueError("insufficient funds")
        self._balance -= amount

    def balance(self):
        return self._balance   # the only sanctioned way to read it


acct = BankAccount(100)
acct.deposit(50)
acct.withdraw(30)
assert acct.balance() == 120
try:
    acct.withdraw(10_000)
    raise AssertionError("withdraw should have raised ValueError")
except ValueError:
    print("withdraw correctly rejected an overdraw; balance is still", acct.balance())

In [ ]:
class Temperature:
    """Stores a temperature in Celsius, refusing physically impossible values."""

    ABSOLUTE_ZERO_C = -273.15

    def __init__(self, celsius):
        if celsius < self.ABSOLUTE_ZERO_C:
            # rejected immediately -- the object is never created in a bad state
            raise ValueError(f"{celsius}C is below absolute zero")
        self._celsius = celsius

    def celsius(self):
        return self._celsius

    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32


t = Temperature(25)
assert round(t.fahrenheit(), 1) == 77.0
try:
    Temperature(-300)
    raise AssertionError("should have rejected an impossible temperature")
except ValueError:
    print("Temperature correctly rejected -300C")

## 3. Inheritance and polymorphism

`Circle` and `Square` written as unrelated classes just happen to share the same shape (`__init__`, `area()`). If you want to loop over a mixed list of shapes and treat them the same way, you need to *name* that shared relationship. **Inheritance** (`class Circle(Shape):`) declares an "is-a" relationship: a `Circle` *is a* `Shape`. Writing `area(self): raise NotImplementedError` on the base class forces every subclass to override it — forgetting to do so fails loudly rather than silently returning nonsense.

**Polymorphism** means the same method call, `shape.area()`, produces different behavior depending on the actual type of `shape` — the calling code never checks "is this a Circle or a Square?"; it simply trusts each object to answer for its own type. Add a new shape later, and the loop below needs zero changes.

In [ ]:
class Shape:
    def area(self):
        raise NotImplementedError   # forces every subclass to override this

class Circle(Shape):        # Circle IS-A Shape
    def __init__(self, r):
        self.r = r
    def area(self):
        return 3.14159 * self.r ** 2

class Square(Shape):        # Square IS-A Shape
    def __init__(self, s):
        self.s = s
    def area(self):
        return self.s ** 2

class Triangle(Shape):      # Triangle IS-A Shape
    def __init__(self, b, h):
        self.b = b
        self.h = h
    def area(self):
        return 0.5 * self.b * self.h


shapes = [Circle(2), Square(3), Triangle(4, 5)]
for shape in shapes:
    # the loop never asks "what kind of shape is this?" -- polymorphism handles it
    print(type(shape).__name__, shape.area())

assert round(Circle(2).area(), 2) == 12.57
assert Square(3).area() == 9
assert Triangle(4, 5).area() == 10.0
print("shape hierarchy checks passed")

**Why this pays off later in the course:** every data structure this term (`Stack`, `Queue`, `LinkedListStack`, ...) follows exactly this pattern — state hidden in `__init__`, behavior exposed as methods. Composition ("has-a") is the default choice when one object simply *contains* another; reach for inheritance only when "is-a" is genuinely, unambiguously true.

In [ ]:
class Engine:
    def start(self):
        return "engine started"

class Car:
    def __init__(self):
        self.engine = Engine()   # composition: Car HAS an Engine, is not one

    def start(self):
        return self.engine.start()


print(Car().start())   # 'engine started'
# Car should NOT inherit from Engine: a car is not a kind of engine.
# If it did, car.start() would be ambiguous about whose start() is meant.

## 4. Big-O: why we count operations, not seconds

Timing code in seconds depends on the machine, the load on it, and the language runtime — not a fair way to compare two approaches in the abstract. Instead we count how the number of basic operations grows as the input size `n` grows, which is what **Big-O** notation describes. Five classes come up constantly, from cheapest to most expensive:

| Class | Meaning | Example |
|---|---|---|
| O(1) | constant, independent of n | dictionary/array lookup by key or index |
| O(log n) | halves the problem each step | binary search on a sorted list |
| O(n) | one pass over every element | linear search, summing a list |
| O(n log n) | a good sort | merge sort (Week 5) |
| O(n²) | a loop nested inside a loop over the same n items | comparing every pair (e.g. checking all-pairs for a shared birthday) |

A concrete illustration: finding a contact by name among 1,000 unsorted contacts. `speed_dial[2]` (dictionary lookup) is O(1) regardless of how many entries exist. Scanning every contact one by one (`find_linear`) is O(n) — double the contacts, double the worst-case work. Binary search on a *sorted* list of contacts is O(log n) — about 10 comparisons even at 1,000 entries, because each comparison halves the remaining range.

In [ ]:
def find_linear(contacts, name):
    # O(n): in the worst case (name missing, or last), every contact is checked once
    for i, c in enumerate(contacts):
        if c == name:
            return i
    return -1

def binary_search(sorted_contacts, name):
    # O(log n): each comparison halves the remaining search range -- requires a SORTED list
    lo, hi = 0, len(sorted_contacts) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if sorted_contacts[mid] == name:
            return mid
        elif sorted_contacts[mid] < name:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

def has_duplicate_birthday(birthdays):
    # O(n^2): a loop nested inside a loop over the same n items
    n = len(birthdays)
    for i in range(n):
        for j in range(i + 1, n):
            if birthdays[i] == birthdays[j]:
                return True
    return False


names = sorted(["Zara", "Amina", "John", "Priya", "Wei", "Fatima"])
assert find_linear(names, "Priya") == names.index("Priya")
assert binary_search(names, "Priya") == names.index("Priya")
assert binary_search(names, "nobody") == -1
assert has_duplicate_birthday(["1-Jan", "5-May", "1-Jan"]) is True
assert has_duplicate_birthday(["1-Jan", "5-May", "9-Sep"]) is False
print("search and duplicate-check functions behave as expected")

We can *see* the O(n) vs O(n²) gap directly, by timing `find_linear` (should scale roughly linearly with `n`) against `has_duplicate_birthday` on a list with no duplicates, its true worst case (should scale much faster than linearly). We do not claim specific numbers in advance — the cell below measures them live with `timeit`, on whatever machine runs it.

In [ ]:
import timeit

for n in (500, 1000, 2000):
    data = [f"item-{i}" for i in range(n)]   # a target that is NOT in the list -> worst case
    t_linear = timeit.timeit(lambda: find_linear(data, "not-present"), number=50)
    t_quadratic = timeit.timeit(lambda: has_duplicate_birthday(data), number=5)
    print(f"n={n:>5}  find_linear (50 runs)={t_linear:.4f}s   "
          f"has_duplicate_birthday (5 runs, worst case)={t_quadratic:.4f}s")

As `n` doubles, `find_linear`'s time roughly doubles too (O(n)); `has_duplicate_birthday`'s time grows noticeably faster than that (closer to O(n²), since doubling `n` roughly quadruples the number of pairs compared).

## Exercises

Work through these in order — later exercises build on ideas from earlier ones. Each exercise gives you a partially-written function or class: fill in the body where you see `# TODO`.

### Exercise 1 — `Rectangle` with encapsulated dimensions

Write a `Rectangle` class that stores `width` and `height` privately (leading underscore), exposes `area()` and `perimeter()` methods, and refuses non-positive dimensions by raising `ValueError` from `__init__`.

Example:
```python
r = Rectangle(4, 5)
r.area()       # 20
r.perimeter()  # 18
Rectangle(-1, 5)  # raises ValueError
```

In [ ]:
class Rectangle:
    """A rectangle with encapsulated, validated width and height."""

    def __init__(self, width, height):
        # TODO: validate width > 0 and height > 0, raising ValueError otherwise;
        # then store them as self._width and self._height
        raise NotImplementedError

    def area(self):
        # TODO: return width * height
        raise NotImplementedError

    def perimeter(self):
        # TODO: return 2 * (width + height)
        raise NotImplementedError

**Self-check — Exercise 1**

In [ ]:
r = Rectangle(4, 5)
assert r.area() == 20
assert r.perimeter() == 18
try:
    Rectangle(-1, 5)
    raise AssertionError("Rectangle should reject a non-positive width")
except ValueError:
    pass
print("\u2705 Exercise 1 passed")

### Exercise 2 — a small vehicle hierarchy (inheritance + polymorphism)

Write a base class `Vehicle` with a method `describe(self)` that raises `NotImplementedError`, and two subclasses:
- `Car(Vehicle)`: `__init__(self, wheels=4)`, `describe()` returns `"Car with 4 wheels"` (using the actual wheel count).
- `Motorcycle(Vehicle)`: `__init__(self)`, `describe()` returns `"Motorcycle with 2 wheels"`.

Example:
```python
vehicles = [Car(), Motorcycle()]
[v.describe() for v in vehicles]  # ['Car with 4 wheels', 'Motorcycle with 2 wheels']
```

In [ ]:
class Vehicle:
    def describe(self):
        raise NotImplementedError

class Car(Vehicle):
    def __init__(self, wheels=4):
        # TODO: store wheels
        raise NotImplementedError

    def describe(self):
        # TODO: return f"Car with {self.wheels} wheels"
        raise NotImplementedError

class Motorcycle(Vehicle):
    def __init__(self):
        # TODO: store wheels = 2
        raise NotImplementedError

    def describe(self):
        # TODO: return f"Motorcycle with {self.wheels} wheels"
        raise NotImplementedError

**Self-check — Exercise 2**

In [ ]:
vehicles = [Car(), Motorcycle(), Car(wheels=6)]
descriptions = [v.describe() for v in vehicles]
assert descriptions == ["Car with 4 wheels", "Motorcycle with 2 wheels", "Car with 6 wheels"]
print("\u2705 Exercise 2 passed")

### Exercise 3 — classify the complexity

Write `classify(fn, sizes)` where `fn` is a one-argument function that takes a list and `sizes` is a list of input sizes to test. For each size `n` in `sizes`, build a list `list(range(n))`, count how many times a marked "basic operation" runs inside `fn` (already instrumented for you below), and return the list of operation counts. Then use the counts you get to decide, by inspection, whether `mystery_function` below is O(1), O(n), or O(n²) — write your answer as a one-line Python comment beneath the cell.

The counting machinery is provided; you only need to complete `classify`.

In [ ]:
counter = {"ops": 0}

def mystery_function(items):
    counter["ops"] = 0
    total = 0
    for i in items:
        for j in items:
            counter["ops"] += 1   # the basic operation being counted
            total += 1
    return total

def classify(fn, sizes):
    """Run fn on list(range(n)) for each n in sizes; return the list of
    operation counts recorded in counter['ops'] after each run.
    """
    # TODO: for each n in sizes, call fn(list(range(n))), then append
    # counter["ops"] to a results list. Return the results list.
    raise NotImplementedError

**Self-check — Exercise 3**

In [ ]:
counts = classify(mystery_function, [2, 4, 8])
assert counts == [4, 16, 64], counts   # n^2 operations each time: 2^2, 4^2, 8^2
print("\u2705 Exercise 3 passed -- counts:", counts)
# mystery_function is O(n^2): counts scale with the square of n (4, 16, 64 for n = 2, 4, 8)

### Exercise 4 (harder) — `Library` composed of `Book` objects

Write a `Book` class storing `title` and `is_checked_out` (starts `False`), with methods `check_out()` (raises `ValueError` if already checked out) and `return_book()` (raises `ValueError` if not currently checked out). Then write a `Library` class that *has a* list of `Book` objects (composition, not inheritance) with methods `add_book(book)`, `check_out(title)` (finds the book by title and checks it out, raising `KeyError` if no such title exists), and `available_titles()` (returns a sorted list of titles that are not checked out).

Example:
```python
lib = Library()
lib.add_book(Book("Fluent Python"))
lib.add_book(Book("The Algorithm Design Manual"))
lib.check_out("Fluent Python")
lib.available_titles()  # ['The Algorithm Design Manual']
```

In [ ]:
class Book:
    def __init__(self, title):
        self.title = title
        self.is_checked_out = False

    def check_out(self):
        # TODO: raise ValueError if already checked out, else set is_checked_out = True
        raise NotImplementedError

    def return_book(self):
        # TODO: raise ValueError if not checked out, else set is_checked_out = False
        raise NotImplementedError


class Library:
    def __init__(self):
        self._books = []   # composition: Library HAS Books, is not a kind of Book

    def add_book(self, book):
        # TODO: append book to self._books
        raise NotImplementedError

    def check_out(self, title):
        # TODO: find the Book with this title in self._books and call check_out() on it;
        # if no book with that title exists, raise KeyError(title)
        raise NotImplementedError

    def available_titles(self):
        # TODO: return a sorted list of titles of books where is_checked_out is False
        raise NotImplementedError

**Self-check — Exercise 4**

In [ ]:
lib = Library()
lib.add_book(Book("Fluent Python"))
lib.add_book(Book("The Algorithm Design Manual"))
lib.check_out("Fluent Python")
assert lib.available_titles() == ["The Algorithm Design Manual"]
try:
    lib.check_out("Fluent Python")
    raise AssertionError("should not allow checking out an already-checked-out book")
except ValueError:
    pass
try:
    lib.check_out("Nonexistent Book")
    raise AssertionError("should raise KeyError for an unknown title")
except KeyError:
    pass
print("\u2705 Exercise 4 passed")

## Quiz

**1. What does `self` refer to inside a method?**
<details><summary>Show answer</summary>The specific object the method was called on. Python passes it automatically as the first argument; you never supply it yourself when calling <code>obj.method(...)</code>.</details>

**2. Why is a leading underscore like `_balance` called a "convention, not a lock"?**
<details><summary>Show answer</summary>Python has no <code>private</code> keyword. Nothing stops code outside the class from writing <code>account._balance = -500</code> directly. The underscore only signals intent to other programmers: "go through the methods, not this attribute." Encapsulation in Python is a social contract enforced by the class's methods, not by the language.</details>

**3. `Dog` and `Animal`; `Car` and `Wheel` — which pair is inheritance ("is-a") and which is composition ("has-a")?**
<details><summary>Show answer</summary><code>Dog(Animal)</code> is inheritance: a dog genuinely is a kind of animal. <code>Car</code> and <code>Wheel</code> is composition: a car <em>has</em> wheels, but a car is not a kind of wheel.</details>

**4. Classify this snippet's complexity: `def f(items): return items[len(items)//2]`**
<details><summary>Show answer</summary>O(1). A single index access, regardless of how large <code>items</code> is — no loop, no dependence on n beyond one arithmetic computation of the index.</details>

## Solutions (try the exercises yourself first!)

Complete, runnable solutions to all four exercises.

In [ ]:
# --- Exercise 1 solution ---
class Rectangle:
    def __init__(self, width, height):
        if width <= 0 or height <= 0:
            raise ValueError("width and height must be positive")
        self._width = width
        self._height = height

    def area(self):
        return self._width * self._height

    def perimeter(self):
        return 2 * (self._width + self._height)


r = Rectangle(4, 5)
assert r.area() == 20 and r.perimeter() == 18
print("Exercise 1 solution verified")

In [ ]:
# --- Exercise 2 solution ---
class Vehicle:
    def describe(self):
        raise NotImplementedError

class Car(Vehicle):
    def __init__(self, wheels=4):
        self.wheels = wheels

    def describe(self):
        return f"Car with {self.wheels} wheels"

class Motorcycle(Vehicle):
    def __init__(self):
        self.wheels = 2

    def describe(self):
        return f"Motorcycle with {self.wheels} wheels"


assert [v.describe() for v in [Car(), Motorcycle()]] == ["Car with 4 wheels", "Motorcycle with 2 wheels"]
print("Exercise 2 solution verified")

In [ ]:
# --- Exercise 3 solution ---
def classify(fn, sizes):
    results = []
    for n in sizes:
        fn(list(range(n)))
        results.append(counter["ops"])
    return results


assert classify(mystery_function, [2, 4, 8]) == [4, 16, 64]
print("Exercise 3 solution verified")

In [ ]:
# --- Exercise 4 solution ---
class Book:
    def __init__(self, title):
        self.title = title
        self.is_checked_out = False

    def check_out(self):
        if self.is_checked_out:
            raise ValueError(f"{self.title} is already checked out")
        self.is_checked_out = True

    def return_book(self):
        if not self.is_checked_out:
            raise ValueError(f"{self.title} was not checked out")
        self.is_checked_out = False


class Library:
    def __init__(self):
        self._books = []

    def add_book(self, book):
        self._books.append(book)

    def check_out(self, title):
        for book in self._books:
            if book.title == title:
                book.check_out()
                return
        raise KeyError(title)

    def available_titles(self):
        return sorted(b.title for b in self._books if not b.is_checked_out)


lib = Library()
lib.add_book(Book("Fluent Python"))
lib.add_book(Book("The Algorithm Design Manual"))
lib.check_out("Fluent Python")
assert lib.available_titles() == ["The Algorithm Design Manual"]
print("Exercise 4 solution verified")

## MTech Extension — duck typing, ABCs, and formal Big-O

**Duck typing.** Polymorphism above worked through a shared base class (`Shape`). Python does not actually require that: if an object has an `area()` method, calling code that expects `.area()` will accept it, base class or not — "if it walks like a duck and quacks like a duck." This is convenient but silent: a typo like `def rea(self)` fails only when `.area()` is finally called, possibly far from where the class was defined.

The `abc` module lets you make the expectation explicit and enforced: `Shape(ABC)` with an `@abstractmethod area(self)` cannot even be *instantiated* unless every abstract method is overridden — the failure moves from "silent, at call time" to "loud, at construction time", which is almost always preferable in a larger codebase.

**Formal Big-O.** Informally we've been saying "the operation count grows like n²." The formal definition: `f(n) = O(g(n))` if there exist constants `c > 0` and `n0` such that `f(n) <= c * g(n)` for all `n >= n0`. This is why constant factors and lower-order terms are dropped: `f(n) = 3n^2 + 100n + 7` is `O(n^2)`, because for large enough `n` the `n^2` term dominates and a suitable `c` (e.g. `c = 4`) absorbs the rest.

In [ ]:
from abc import ABC, abstractmethod

class ShapeABC(ABC):
    @abstractmethod
    def area(self):
        ...

class IncompleteShape(ShapeABC):
    pass   # forgot to override area()

try:
    ShapeABC()   # TypeError: can't instantiate abstract class
    raise AssertionError("should not be able to instantiate an ABC directly")
except TypeError:
    print("ShapeABC correctly refuses direct instantiation")

try:
    IncompleteShape()   # TypeError: area() was never overridden
    raise AssertionError("should not be able to instantiate without overriding area()")
except TypeError:
    print("IncompleteShape correctly refuses instantiation without area()")


class Duck:
    def area(self):
        return 0   # a Duck has no ABC ancestry at all, but still has .area()

def total_area(shapes):
    # accepts ANYTHING with .area(), ABC-registered or not -- duck typing
    return sum(s.area() for s in shapes)

print(total_area([Duck(), Duck()]))   # 0 -- duck typing accepted it with no shared base class

**Discussion point for MTech:** verify the formal O(n²) claim on `f(n) = 3n^2 + 100n + 7` numerically — pick `c = 4` and find an `n0` beyond which `f(n) <= 4 * n^2` holds, then confirm it stays true as `n` grows further.

In [ ]:
def f(n):
    return 3 * n**2 + 100 * n + 7

c = 4
for n in (1, 10, 34, 35, 100, 1000):
    bound = c * n**2
    print(f"n={n:>5}  f(n)={f(n):>12}  {c}*n^2={bound:>12}  f(n) <= bound: {f(n) <= bound}")
# f(n) <= 4*n^2 fails for very small n (the lower-order terms dominate briefly)
# but holds for all n from n0=35 onward -- exactly what the formal O(n^2) definition requires